## 실습 6 — 유압 계통 데이터 해석

사용 파일
- `03-01_유압·열설비_신호_계통태그목록.csv`
- `03-01_유압·열설비_신호_유압운전.csv`


In [6]:
import os
import pandas as pd

pd.set_option("display.max_columns", None)


In [7]:
tag_path = os.path.join("..", "Data", "03-01_유압·열설비_신호_계통태그목록.csv")
hyd_path = os.path.join("..", "Data", "03-01_유압·열설비_신호_유압운전.csv")

tags = pd.read_csv(tag_path, encoding="utf-8")
hyd_df = pd.read_csv(hyd_path, encoding="utf-8")

print(hyd_df.head().to_markdown(index=False))


| date       |   HYD01_PRESS_PUMP |   HYD01_PRESS_FILT_IN |   HYD01_PRESS_FILT_OUT |   HYD01_FLOW |   HYD01_OILTEMP |   HYD01_LEVEL |   HYD01_PUMP_CURRENT |
|:-----------|-------------------:|----------------------:|-----------------------:|-------------:|----------------:|--------------:|---------------------:|
| 2026-01-01 |                152 |                   150 |                  147.5 |        118   |            42   |            88 |                 31.5 |
| 2026-01-02 |                153 |                   151 |                  148.5 |        118.5 |            42   |            88 |                 31.5 |
| 2026-01-03 |                152 |                   150 |                  147.5 |        117.5 |            42.5 |            88 |                 31   |
| 2026-01-04 |                151 |                   149 |                  146   |        118   |            42.5 |            88 |                 32   |
| 2026-01-05 |                153 |                   151 

### Step 1. 주요 태그 확인

HYD로 시작하는 태그를 확인하고 아래 항목을 정리하세요. (유압 계통)  
Python으로 진행해주세요.


In [8]:
hyd_tags = tags[tags["tag"].str.startswith("HYD")]
step1_table = hyd_tags[["tag", "physical_qty", "unit", "circuit_position"]]

print(step1_table.to_markdown(index=False))


| tag                  | physical_qty   | unit   | circuit_position    |
|:---------------------|:---------------|:-------|:--------------------|
| HYD01_PRESS_PUMP     | 압력           | bar    | 펌프 토출부         |
| HYD01_PRESS_FILT_IN  | 압력           | bar    | 필터 전단           |
| HYD01_PRESS_FILT_OUT | 압력           | bar    | 필터 후단           |
| HYD01_DP_FILTER      | 차압           | bar    | 필터 전후단 계산값  |
| HYD01_FLOW           | 유량           | L/min  | 펌프 토출 배관      |
| HYD01_OILTEMP        | 온도           | degC   | 탱크 내부           |
| HYD01_LEVEL          | 유면           | %      | 탱크 유면계         |
| HYD01_PUMP_CURRENT   | 전류           | A      | 펌프 제어반         |
| HYD01_VALVE_CMD      | 개도           | %      | 방향 제어 밸브 지령 |
| HYD01_VALVE_FB       | 개도           | %      | 방향 제어 밸브 실제 |


### Step 2. 정상 구간과 최근 구간 비교

아래 5개 컬럼을 사용하세요.

- `HYD01_PRESS_PUMP`
- `HYD01_FLOW`
- `HYD01_OILTEMP`
- `HYD01_LEVEL`
- `HYD01_PUMP_CURRENT`

Python으로 다음 값을 구하세요.

1. 첫 30일 평균
2. 마지막 10일 평균


In [9]:
normal_30 = hyd_df.head(30)
recent_10 = hyd_df.tail(10)

rows = []

normal_mean = normal_30["HYD01_PRESS_PUMP"].mean()
recent_mean = recent_10["HYD01_PRESS_PUMP"].mean()
rows.append(["펌프 압력", normal_mean, recent_mean, "감소"])

normal_mean = normal_30["HYD01_FLOW"].mean()
recent_mean = recent_10["HYD01_FLOW"].mean()
rows.append(["유량", normal_mean, recent_mean, "감소"])

normal_mean = normal_30["HYD01_OILTEMP"].mean()
recent_mean = recent_10["HYD01_OILTEMP"].mean()
rows.append(["유온", normal_mean, recent_mean, "증가"])

normal_mean = normal_30["HYD01_LEVEL"].mean()
recent_mean = recent_10["HYD01_LEVEL"].mean()
rows.append(["유면", normal_mean, recent_mean, "유지"])

normal_mean = normal_30["HYD01_PUMP_CURRENT"].mean()
recent_mean = recent_10["HYD01_PUMP_CURRENT"].mean()
rows.append(["펌프 전류", normal_mean, recent_mean, "증가"])

step2_table = pd.DataFrame(rows, columns=["물리량", "정상 30일 평균", "최근 10일 평균", "변화 방향"])
step2_table["정상 30일 평균"] = step2_table["정상 30일 평균"].round(2)
step2_table["최근 10일 평균"] = step2_table["최근 10일 평균"].round(2)

print(step2_table.to_markdown(index=False))


| 물리량    |   정상 30일 평균 |   최근 10일 평균 | 변화 방향   |
|:----------|-----------------:|-----------------:|:------------|
| 펌프 압력 |           152.3  |           149.5  | 감소        |
| 유량      |           117.95 |           116.95 | 감소        |
| 유온      |            42.35 |            48.35 | 증가        |
| 유면      |            88    |            88    | 유지        |
| 펌프 전류 |            31.45 |            32.45 | 증가        |


질문  
최근 구간에서 어떤 변화가 함께 나타났는지 한 문장으로 정리하세요.

→ 펌프 압력과 유량은 줄어들고, 유온과 펌프 전류는 증가했다. 유면은 그대로라서 밖으로 새는 문제만으로 보기는 어렵다.


### Step 3. 필터 차압 확인

필터 전단 압력과 후단 압력을 이용해 차압을 계산하세요.  
차압은 csv 컬럼으로 관리하고 있지 않아서 직접 계산해야합니다.  
Python으로 진행해주세요.

차압 = 필터 전단 압력 - 필터 후단 압력


In [10]:
hyd_df["DP_FILTER"] = hyd_df["HYD01_PRESS_FILT_IN"] - hyd_df["HYD01_PRESS_FILT_OUT"]

rows = []

period = hyd_df.iloc[0:30]
start_dp = period["DP_FILTER"].iloc[0]
end_dp = period["DP_FILTER"].iloc[-1]
rows.append(["1~30일", start_dp, end_dp, end_dp - start_dp])

period = hyd_df.iloc[30:60]
start_dp = period["DP_FILTER"].iloc[0]
end_dp = period["DP_FILTER"].iloc[-1]
rows.append(["31~60일", start_dp, end_dp, end_dp - start_dp])

period = hyd_df.iloc[60:90]
start_dp = period["DP_FILTER"].iloc[0]
end_dp = period["DP_FILTER"].iloc[-1]
rows.append(["61~90일", start_dp, end_dp, end_dp - start_dp])

step3_table = pd.DataFrame(rows, columns=["구간", "시작 차압", "종료 차압", "증가폭"])

print(step3_table.to_markdown(index=False))


| 구간    |   시작 차압 |   종료 차압 |   증가폭 |
|:--------|------------:|------------:|---------:|
| 1~30일  |         2.5 |           5 |      2.5 |
| 31~60일 |         2.5 |           6 |      3.5 |
| 61~90일 |         2.5 |           7 |      4.5 |




질문  
차압 증가폭이 구간별로 어떻게 달라지는지 적으세요.

→ 차압 증가폭이 2.5에서 3.5, 4.5로 점점 커진다.


### Step 4. 최종 해석

1. 유면은 그대로인데 유온과 전류가 증가했다면 단순한 외부 누유라고 볼 수 있을까요?

→ 단순한 외부 누유라고 보기는 어렵다. 외부로 새면 유면이 같이 내려가야 하는데, 유면은 그대로이기 때문이다.

2. 필터 교체 후 차압은 다시 낮아지지만, 다음 구간에서 차압 증가폭이 더 커진다면 무엇을 의심할 수 있을까요?

→ 필터만의 문제라기보다 유압유 오염이나 내부 마모로 이물질이 계속 늘어나는 상황을 의심할 수 있다.

3. 전체 결과를 2~3문장으로 정리하세요.

→ 최근 구간에서는 펌프 압력과 유량은 감소했고, 유온과 펌프 전류는 증가했다.  
→ 유면은 그대로라서 단순 외부 누유보다는 내부 누설이나 펌프 효율 저하 가능성이 더 크다. 차압 증가폭도 점점 커지므로 유압유 오염이나 필터 막힘이 빨라지는 상태도 같이 의심할 수 있다.
